# Notebook 03: Normal-Lung Density Modeling & Proxy Diagnostic Evaluation

This notebook walks through:
1. Fitting Gaussian Mixture Model (GMM) and RealNVP Normalizing Flow on healthy control embeddings.
2. Evaluating patch anomaly scores $s(x) = -\log p(z)$.
3. Aggregating patch scores into patient-level severity indices.
4. Evaluating proxy AUROC and AUPRC against radiology report labels with 95% bootstrap confidence intervals.

> **MANDATORY NOTICE:** All diagnostic classifications in this notebook are strictly **REPORT-LABEL PROXIES, not clinical COPD diagnosis**.


In [ ]:
import numpy as np
import pandas as pd
from copd_anomaly.anomaly.density import GaussianMixtureDensity, PatchAnomalyScorer, RealNVPFlowDensity
from copd_anomaly.anomaly.aggregation import ScoreAggregator, extract_patient_anomaly_summary
from copd_anomaly.evaluation.metrics import ModelEvaluator, PROXY_DISCLAIMER

print(PROXY_DISCLAIMER)


## 1. Density Estimation on Normal Healthy Lung


In [ ]:
rng = np.random.RandomState(42)
z_normal = rng.normal(loc=0.0, scale=1.0, size=(300, 32)).astype(np.float32)
z_abnormal = rng.normal(loc=4.5, scale=1.5, size=(50, 32)).astype(np.float32)

gmm = GaussianMixtureDensity(n_components=6, covariance_type='diag', random_state=42)
gmm.fit(z_normal)

scorer = PatchAnomalyScorer(gmm)
scores_norm = scorer.score_patches(z_normal[:50])
scores_abn = scorer.score_patches(z_abnormal)

print(f'Normal patches anomaly score:   {scores_norm.mean():.2f} ± {scores_norm.std():.2f}')
print(f'Abnormal patches anomaly score: {scores_abn.mean():.2f} ± {scores_abn.std():.2f}')


## 2. Proxy AUROC & AUPRC with Bootstrap 95% Confidence Intervals


In [ ]:
evaluator = ModelEvaluator()
y_true = np.array([0] * 50 + [1] * 50)
y_score = np.concatenate([scores_norm, scores_abn])

metrics = evaluator.evaluate_proxy_auc(y_true, y_score, n_bootstraps=500)
print('Proxy Diagnostic Evaluation Results:')
print(f'  AUROC: {metrics["auroc"]:.3f} (95% CI: [{metrics["auroc_ci95_low"]:.3f}, {metrics["auroc_ci95_high"]:.3f}])')
print(f'  AUPRC: {metrics["auprc"]:.3f} (95% CI: [{metrics["auprc_ci95_low"]:.3f}, {metrics["auprc_ci95_high"]:.3f}])')
print(f'  Optimal Sensitivity: {metrics["sensitivity_at_opt"]:.1%}')
print(f'  Optimal Specificity: {metrics["specificity_at_opt"]:.1%}')
